# Customer Satisfaction and Complaint Analysis
**Tools:** Python (pandas), Excel, Power BI / Tableau-ready data  
**Goal:** Categorise customer complaints, track resolution rate and top recurring issues, and turn them into short recommendations for reducing repeat complaints.

> **Data note:** This notebook is designed for the **CFPB Consumer Complaint Database** (public data, https://www.consumerfinance.gov/data-research/consumer-complaints/).  
> - If `data/complaints_public.csv` exists, the notebook uses it (the CFPB CSV export, original column names).  
> - Otherwise it generates a **synthetic sample** with the same structure so the notebook runs end-to-end.  
> Only describe the project as built on *public data* once you have run it on the real file.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

np.random.seed(7)
DATA = Path("data"); DATA.mkdir(exist_ok=True)
PUBLIC_CSV = DATA / "complaints_public.csv"
COLS = ["complaint_id", "date_received", "product", "issue", "company_response", "timely_response", "disputed"]

## 1. Load data (real CFPB file if present, otherwise synthetic sample)

In [ ]:
if PUBLIC_CSV.exists():
    raw = pd.read_csv(PUBLIC_CSV)
    df = raw.rename(columns={"Complaint ID": "complaint_id", "Date received": "date_received", "Product": "product",
                             "Issue": "issue", "Company response to consumer": "company_response",
                             "Timely response?": "timely_response", "Consumer disputed?": "disputed"}).reindex(columns=COLS)
    SOURCE = "CFPB public data"
else:
    issues = {
      "Credit card": ["Billing dispute", "Fraud or unauthorized charge", "Problem with rewards", "Late fee"],
      "Bank account": ["Account closed without notice", "Deposit held", "Unauthorized transaction", "Poor customer service"],
      "Loan": ["Payment processing delay", "Incorrect interest charged", "Difficulty getting information"],
      "Mobile payments": ["Failed transaction", "Refund not received", "Account access problem"],
    }
    product_p = [.35, .30, .20, .15]
    n = 2400
    products = np.random.choice(list(issues), n, p=product_p)
    dates = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 630, n), unit="D")
    iss = []
    for p, d in zip(products, dates):
        opts = issues[p]
        w = np.array([3.0 if i == 0 else 1.5 if i == 1 else 1.0 for i in range(len(opts))])
        if p == "Mobile payments" and d >= pd.Timestamp("2026-03-01"):    # emerging problem in later months
            w[1] *= 2.5
        iss.append(np.random.choice(opts, p=w / w.sum()))
    df = pd.DataFrame({
        "complaint_id": np.arange(100000, 100000 + n), "date_received": dates, "product": products, "issue": iss,
        "company_response": np.random.choice(
            ["Closed with explanation", "Closed with monetary relief", "Closed with non-monetary relief", "In progress"],
            n, p=[.62, .14, .12, .12]),
        "timely_response": np.random.choice(["Yes", "No"], n, p=[.96, .04]),
        "disputed": np.random.choice(["Yes", "No"], n, p=[.18, .82])})
    SOURCE = "synthetic sample"

df["date_received"] = pd.to_datetime(df["date_received"])
df["month"] = df["date_received"].dt.to_period("M").dt.to_timestamp()
df["closed"] = df["company_response"].str.startswith("Closed")
df["relief"] = df["company_response"].str.contains("relief", case=False) & ~df["company_response"].str.contains("without", case=False)
print(f"Source: {SOURCE} | rows: {len(df):,}")
df.head()

## 2. Clean and categorise

In [ ]:
df = df.dropna(subset=["product", "issue"]).drop_duplicates("complaint_id")
df["product"] = df["product"].str.strip()
df["issue"] = df["issue"].str.strip()
df.isna().sum()

## 3. Resolution KPIs

In [ ]:
total = len(df)
kpi = {
  "Total complaints": total,
  "Closed (resolution rate %)": round(100 * df["closed"].mean(), 1),
  "Closed with relief (% of closed)": round(100 * df.loc[df["closed"], "relief"].mean(), 1),
  "Timely response %": round(100 * (df["timely_response"] == "Yes").mean(), 1),
}
if df["disputed"].notna().any():
    kpi["Disputed % (of answered)"] = round(100 * (df["disputed"] == "Yes").sum() / df["disputed"].notna().sum(), 1)
pd.Series(kpi, name="value").to_frame()

In [ ]:
by_product = (df.groupby("product").agg(complaints=("complaint_id", "count"),
                                         resolution_rate_pct=("closed", lambda s: round(100 * s.mean(), 1)),
                                         timely_pct=("timely_response", lambda s: round(100 * (s == "Yes").mean(), 1)))
              .sort_values("complaints", ascending=False))
by_product

## 4. Top recurring issues (Pareto)

In [ ]:
top = df["issue"].value_counts().head(10).rename_axis("issue").reset_index(name="complaints")
top["share_pct"] = (100 * top["complaints"] / total).round(1)
top["cumulative_pct"] = top["share_pct"].cumsum().round(1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top["issue"][::-1], top["complaints"][::-1], color="#4C78A8")
ax.set_title("Top recurring complaint issues"); ax.set_xlabel("Complaints")
plt.tight_layout(); plt.savefig(DATA / "top_issues.png", dpi=150); plt.show()
top

## 5. Trend: are top issues growing?

In [ ]:
top5 = top["issue"].head(5).tolist()
trend = df[df["issue"].isin(top5)].pivot_table(index="month", columns="issue", values="complaint_id", aggfunc="count").fillna(0)
trend.plot(figsize=(11, 4.5), marker="o"); plt.title("Monthly complaints - top 5 issues")
plt.ylabel("Complaints"); plt.tight_layout(); plt.savefig(DATA / "issue_trend.png", dpi=150); plt.show()

# growth: last 3 months vs the 3 months before
m = sorted(df["month"].unique())
last3, prev3 = m[-3:], m[-6:-3]
g = (df[df["month"].isin(last3)]["issue"].value_counts()
     .sub(df[df["month"].isin(prev3)]["issue"].value_counts(), fill_value=0).sort_values(ascending=False))
g.head(5).rename("change_in_complaints (last 3 mo vs prior 3)").to_frame()

## 6. Resolution quality by issue

In [ ]:
quality = (df.groupby("issue").agg(complaints=("complaint_id", "count"),
                                    resolution_rate_pct=("closed", lambda s: round(100 * s.mean(), 1)),
                                    relief_rate_pct=("relief", lambda s: round(100 * s.mean(), 1)))
           .sort_values("complaints", ascending=False).head(10))
quality

## 7. Recommendations to reduce repeat complaints

In [ ]:
RECO = {
 "Billing dispute": "Show itemised charges clearly in statements and send a proactive alert before disputed charges post.",
 "Fraud or unauthorized charge": "Add real-time transaction alerts and a one-tap 'not me' report with instant card block.",
 "Refund not received": "Give customers a refund tracker with a reference number and send status updates at each stage.",
 "Failed transaction": "Auto-reverse failed payments within a fixed window and notify the customer immediately.",
 "Payment processing delay": "Publish cut-off times and send confirmation once a payment is processed.",
 "Account closed without notice": "Send advance notice and a reason before closing accounts; add a reinstatement path.",
 "Deposit held": "Explain hold reasons and release dates upfront at the time of deposit.",
 "Poor customer service": "Add first-contact resolution targets, call-back guarantees and agent coaching on empathy.",
}
for i, row in top.head(3).iterrows():
    print(f"{i+1}. {row['issue']} ({row['share_pct']}% of complaints)")
    print("   ->", RECO.get(row["issue"], "Review the process behind this issue, add a clear customer-facing FAQ and set a first-contact resolution target."))

## 8. Export for Power BI / Tableau

In [ ]:
df.to_csv(DATA / "complaints_clean.csv", index=False)
top.to_csv(DATA / "top_issues.csv", index=False)
by_product.reset_index().to_csv(DATA / "by_product.csv", index=False)
trend.reset_index().to_csv(DATA / "issue_trend.csv", index=False)
try:
    with pd.ExcelWriter(DATA / "complaint_analysis.xlsx", engine="openpyxl") as xw:
        df.to_excel(xw, sheet_name="Complaints", index=False)
        top.to_excel(xw, sheet_name="Top_Issues", index=False)
        by_product.reset_index().to_excel(xw, sheet_name="By_Product", index=False)
        quality.reset_index().to_excel(xw, sheet_name="Issue_Quality", index=False)
    print("Saved data/complaint_analysis.xlsx")
except ModuleNotFoundError:
    print("Install openpyxl to create the Excel file; CSVs were saved.")

## Dashboard layout (Power BI / Tableau)
- **KPI cards:** total complaints, resolution rate, timely response %, disputed %
- **Bar chart:** top recurring issues (from `top_issues.csv`)
- **Line chart:** monthly trend of top issues (from `issue_trend.csv`)
- **Table/matrix:** product x resolution rate
- **Text box:** the 3 recommendations above
- **Slicers:** product, month